#### Fashion MNIST

In [ ]:
# function 파일 오류 수정 후 런타임을 재시작하지 않고 수정내용을 반영하는 방법
# import importlib
# import multiclass_functions
# importlib.reload(multiclass_functions)

from multiclass_functions import *

In [2]:
# # function 파일 오류 수정 후 런타임을 재시작하지 않고 수정내용을 반영하는 방법
import importlib
import multiclass_functions
importlib.reload(multiclass_functions)

from multiclass_functions import *
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms, datasets
# datasets:  이미지 파일과 정답 레이블을 관리하고, 모델 학습에 사용할 수 있는 Dataset 객체를 생성하는 모듈
# transforms: 모델이 학습할 수 있도록 이미지의 크기, 자료형, 차원, 픽셀값 등을 변환하는 모듈입니다. 데이터 증강(Data Augmentation)에도 사용됩니다.
import matplotlib.pyplot as plt
import numpy as np
import os
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [10]:
# Kaggle에서 fashionmnist 다운로드
import os
import kagglehub

data_path = os.path.join(os.getcwd(), "data")
raw_path = os.path.join(data_path, "FashionMNIST", "raw")

path = kagglehub.dataset_download(
    "zalando-research/fashionmnist",
    output_dir=raw_path
)

100%|██████████| 68.8M/68.8M [00:03<00:00, 22.6MB/s]

Extracting files...


In [15]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

train_DS = datasets.FashionMNIST(root=data_path, train=True, transform=transform)
test_DS = datasets.FashionMNIST(root=data_path, train=False, transform=transform)

In [16]:
BATCH_SIZE = 32
LR = 1e-2
EPOCH = 15
criterion = nn.CrossEntropyLoss()
new_model_train = True

model_type = 'CNN'
dataset = 'FashionMNIST'

save_dir = 'results'
os.makedirs(save_dir, exist_ok=True)

save_model_path = f'{save_dir}/{model_type}_{dataset}.pt'

In [17]:
train_DL = DataLoader(train_DS, batch_size= BATCH_SIZE, shuffle=True)
test_DL = DataLoader(test_DS, batch_size= BATCH_SIZE, shuffle=True)

In [18]:
print(train_DS)
print(test_DS)
print(test_DS.classes)
print(test_DS.class_to_idx)

Dataset FashionMNIST
    Number of datapoints: 60000
    Root location: c:\Users\mega\KYJ\PyTorch\05.CNN\data
    Split: Train
    StandardTransform
Transform: Compose(
               Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
           )
Dataset FashionMNIST
    Number of datapoints: 10000
    Root location: c:\Users\mega\KYJ\PyTorch\05.CNN\data
    Split: Test
    StandardTransform
Transform: Compose(
               Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
           )
['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
{'T-shirt/top': 0, 'Trouser': 1, 'Pullover': 2, 'Dress': 3, 'Coat': 4, 'Sandal': 5, 'Shirt': 6, 'Sneaker': 7, 'Bag': 8, 'Ankle boot': 9}


In [20]:
x_batch, y_batch = next(iter(train_DL))
print(x_batch.shape)

torch.Size([32, 1, 64, 64])


In [ ]:
class CNN(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1,8,3, padding=1),    # input(32,1,28,28)
            nn.BatchNorm2d(8),
            nn.ReLU()
            )
        self.maxpool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Sequential(
            nn.Conv2d(8,16,3),
            nn.BatchNorm2d(16),
            nn.ReLU()
            )
        self.maxpool2 = nn.MaxPool2d(2)
        self.conv3 = nn.Sequential(
            nn.Conv2d(16,32,3),
            nn.BatchNorm2d(32),
            nn.ReLU()
            )
        self.maxpool3 = nn.MaxPool2d(2)
        self.fc = nn.Linear(32*2*2, 10)

    def forward(self, x, *args, **kwargs):
        x = self.conv1(x)
        x = self.maxpool1(x)
        x = self.conv2(x)
        x = self.maxpool2(x)
        x = self.conv3(x)
        x = self.maxpool3(x)
        x = torch.flatten(x, start_dim=1)
        x = self.fc(x)
        return x